# CoRe-TFM — Simple JMLR Robustness Run

This is the deliberately simple notebook. It runs **one complete robustness variant at a time** using the proven Q1 benchmark implementation.

There is no experiment queue, no nested Jupyter kernel, no AMD-specific setup, and no automatic multi-seed orchestration. Change only the values in **Cell 2**, then run all. Completed folds are checkpointed to Google Drive, so rerunning the same configuration resumes the run.

Use a Colab GPU runtime and create a Colab Secret named `TABPFN_TOKEN`.


## Cell 1 — get the latest repository and minimal bootstrap dependencies

In [ ]:
from pathlib import Path
import json, os, subprocess, sys, traceback

from google.colab import drive
drive.mount('/content/drive')

ROOT = Path('/content/core-tfm')
if not (ROOT/'.git').exists():
    subprocess.run(['git','clone','https://github.com/bnssaanirudh/core-tfm.git',str(ROOT)], check=True)
else:
    subprocess.run(['git','fetch','origin'], cwd=ROOT, check=True)
    subprocess.run(['git','checkout','main'], cwd=ROOT, check=True)
    subprocess.run(['git','reset','--hard','origin/main'], cwd=ROOT, check=True)

HEAD = subprocess.check_output(['git','rev-parse','HEAD'], cwd=ROOT, text=True).strip()
print('Repository HEAD:', HEAD)

subprocess.run([
    sys.executable,'-m','pip','install','-q','-e','.[test]','pyyaml'
], cwd=ROOT, check=True)

SRC = str(ROOT/'src')
if SRC not in sys.path:
    sys.path.insert(0,SRC)
os.chdir(ROOT)

from core_tfm.robustness_runner import (
    load_notebook, patch_q1_notebook, code_cells_through_shard_12e,
    fold_result_status, write_complete_marker,
)
print('Bootstrap complete.')


## Cell 2 — CONFIGURATION

Normally change only `SEED` and `TRAIN_LIMIT`. Keep the same values when resuming an interrupted run.

Suggested multi-seed runs: `11, 23, 42, 71, 101` with `TRAIN_LIMIT=256`.

Suggested context-size runs: `TRAIN_LIMIT` in `64, 128, 256, 512, 1024`, using seeds `23, 42, 71`.


In [ ]:
SEED = 11
TRAIN_LIMIT = 256
TEST_LIMIT = 128
SHARD_MINUTES = 25

# Give different experiments different run IDs automatically.
RUN_ID = f'simple_runs/seed_{SEED}_train_{TRAIN_LIMIT}'
OUTPUT_ROOT = Path('/content/drive/MyDrive/CoRe_TFM_Q1/core_tfm_jmlr_simple_v1')
RUN_DIR = OUTPUT_ROOT / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)

print('RUN_ID      :', RUN_ID)
print('SEED        :', SEED)
print('TRAIN_LIMIT :', TRAIN_LIMIT)
print('TEST_LIMIT  :', TEST_LIMIT)
print('RUN_DIR     :', RUN_DIR)


## Cell 3 — GPU and configuration preflight

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU is visible. In Colab choose Runtime → Change runtime type → GPU.')
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__)

TEMPLATE = ROOT/'notebooks'/'CoRe_TFM_Q1_FAST_COMPLETE_256_Colab.ipynb'
template = load_notebook(TEMPLATE)
patched = patch_q1_notebook(
    template,
    run_id=RUN_ID,
    seed=SEED,
    train_limit=TRAIN_LIMIT,
    test_limit=TEST_LIMIT,
    drive_base=str(OUTPUT_ROOT),
    session_minutes=SHARD_MINUTES,
    shard_minutes=SHARD_MINUTES,
    disable_controlled_replications=True,
    disable_selection_ablations=True,
    disable_validation_sensitivity=True,
)
engine_cells = code_cells_through_shard_12e(patched)
print('Q1 engine cells through 12E:', len(engine_cells))
print('PRECHECK PASS')

before = fold_result_status(RUN_DIR)
print('Existing progress:')
print(json.dumps(before, indent=2))


## Cell 4 — RUN / RESUME

This is the expensive cell. The original Q1 setup, authentication, dataset preparation, model factories, fold evaluator, and shards 12A–12E execute in order. Each completed fold is saved immediately to Drive.


In [ ]:
current = fold_result_status(RUN_DIR)
if current.get('complete'):
    print('This variant is already complete. Nothing to run.')
else:
    engine_ns = {'__name__':'__core_tfm_simple_run__'}
    for i, source in enumerate(engine_cells, 1):
        first = source.lstrip().splitlines()[0] if source.strip() else '<empty>'
        print(f'\n===== ENGINE CELL {i}/{len(engine_cells)}: {first} =====', flush=True)
        try:
            exec(compile(source, f'<core_tfm_engine_{i}>', 'exec'), engine_ns, engine_ns)
        except Exception:
            print(f'FAILED IN ENGINE CELL {i}: {first}')
            traceback.print_exc()
            raise

after = fold_result_status(RUN_DIR)
print('\nCURRENT VARIANT STATUS:')
print(json.dumps(after, indent=2))
if after.get('last_failure'):
    print('\nLAST RECORDED FOLD FAILURE:')
    print(json.dumps(after['last_failure'], indent=2))
if after.get('complete') and not (RUN_DIR/'COMPLETE.json').exists():
    marker = write_complete_marker(RUN_DIR, {
        'execution_platform':'google_colab_cuda',
        'source_commit':HEAD,
        'seed':SEED,
        'requested_train_limit':TRAIN_LIMIT,
        'requested_test_limit':TEST_LIMIT,
    })
    print('COMPLETE MARKER:', marker)


## Cell 5 — inspect results

In [ ]:
import pandas as pd

status = fold_result_status(RUN_DIR)
print(json.dumps(status, indent=2))

fold_file = RUN_DIR/'fold_results.csv'
if fold_file.exists() and fold_file.stat().st_size > 0:
    folds = pd.read_csv(fold_file)
    print('Rows:', len(folds))
    print('Completed fold cells:', folds[['dataset','model','fold']].drop_duplicates().shape[0], '/ 150')
    display(
        folds.groupby(['model','method'],as_index=False)['joint_nll'].mean()
        .sort_values(['model','joint_nll'])
    )
else:
    print('No fold results have been written yet.')


## Cell 6 — what to do next

If status is incomplete because the Colab session stopped, rerun this notebook **with exactly the same Cell 2 configuration**. Completed folds are skipped.

After one variant reaches 1200 rows / 150 fold cells, change Cell 2 to the next seed or context size and run again. Do not combine partial variants under the same `RUN_ID`.
